# RamanBench — Contributing a New Dataset

This notebook walks through the process of contributing a new Raman spectroscopy
dataset to the RamanBench ecosystem.

## Ecosystem

| Resource | Link |
|---|---|
| **raman-data** (dataset package) | [GitHub](https://github.com/ml-lab-htw/raman_data) · [PyPI](https://pypi.org/project/raman-data/) |
| **raman-bench** (benchmark) | [GitHub](https://github.com/ml-lab-htw/RamanBench) · [PyPI](https://pypi.org/project/raman-bench/) |
| **Live Leaderboard** | [HuggingFace](https://huggingface.co/spaces/ml-lab-htw/RamanBench) |
| **Paper** (NeurIPS 2026) | [arXiv TBD](https://arxiv.org/abs/TBD) |

## Overview

The contribution process has three steps:

1. **Prepare your data** — upload to HuggingFace Datasets or Zenodo (CC BY 4.0)
2. **Add a loader** — open a PR in [raman-data](https://github.com/ml-lab-htw/raman_data)
3. **Request inclusion** — open an issue in [RamanBench](https://github.com/ml-lab-htw/RamanBench/issues/new?template=dataset_submission.md)

## Step 1: Prepare your data

### Required format

Your dataset should be structured as a Pandas DataFrame:
- **Rows** = samples (one spectrum per row)
- **Columns** = wavenumber values (floats, in cm⁻¹)
- **Last column(s)** = target values (regression) or labels (classification)

### Minimum requirements

| Requirement | Details |
|---|---|
| Min samples | ≥ 20 total (≥ 9 per class for classification) |
| License | CC BY 4.0 or more permissive |
| Citation | Published paper or preprint with DOI |
| Instrument | Document excitation wavelength, instrument model, spectral range |

In [ ]:
import numpy as np
import pandas as pd

# Example: create a dataset from scratch
rng = np.random.default_rng(42)
n_samples, n_wavenumbers = 100, 500
wavenumbers = np.linspace(400, 2000, n_wavenumbers)

# Spectra as rows, wavenumbers as column names
spectra_df = pd.DataFrame(
    rng.standard_normal((n_samples, n_wavenumbers)),
    columns=wavenumbers.astype(str),
)
# Add target column(s)
spectra_df['concentration_mM'] = rng.uniform(0, 100, n_samples)

print(spectra_df.shape)
spectra_df.head()

## Step 2: Upload to HuggingFace Datasets

```python
from datasets import Dataset

hf_dataset = Dataset.from_pandas(spectra_df)
hf_dataset.push_to_hub('your-username/my_raman_dataset')
```

Or upload to Zenodo as a CSV/Parquet file.

## Step 3: Add a loader to raman-data

Fork [ml-lab-htw/raman_data](https://github.com/ml-lab-htw/raman_data) and add an entry:

In `raman_data/loaders/HuggingFaceLoader.py`:
```python
DATASETS = {
    # ... existing datasets ...
    'my_compound_concentration': DatasetInfo(
        name='My Compound Raman Dataset',
        task_type=TASK_TYPE.Regression,
        application_type=APPLICATION_TYPE.Chemical,
        source='your-username/my_raman_dataset',
        license='CC BY 4.0',
        citation='Author et al. (2026). doi:10.xxxx/xxxx',
    ),
}
```

## Step 4: Verify the loader works

```python
from raman_data import raman_data

dataset = raman_data('my_compound_concentration')
print(dataset.spectra.shape)
print(dataset.target_names)
```

## Step 5: Request inclusion in RamanBench

Open an issue using the [dataset submission template](https://github.com/ml-lab-htw/RamanBench/issues/new?template=dataset_submission.md).

Once your raman-data PR is merged and a new release is published, we'll add
your dataset to `configs/datasets/` and evaluate all baseline models on it.

## Existing new datasets

For inspiration, see [NEW_DATASETS.md](../NEW_DATASETS.md) which documents
17 datasets released alongside RamanBench v0.1, including fermentation monitoring,
metabolite analysis, and gasoline characterisation datasets.